# Linear Discriminant Analysis (LDA) for Phishing Website Detection

**Setup:** `class` label `-1` = phishing (positive class), `+1` = legitimate. `Index` is excluded. Identical feature vectors are assigned to the same group so duplicates cannot cross the holdout or CV folds. A stratified grouped 5-fold split provides an approximately 80/20 holdout; model selection uses grouped 5-fold CV on training data only. The holdout is evaluated once.


## 1. Upload the dataset


In [ ]:
from google.colab import files
from pathlib import Path

uploaded = files.upload()
csv_names = [n for n in uploaded if n.lower().endswith('.csv')]
if not csv_names:
    raise FileNotFoundError('No .csv file was uploaded. Upload the dataset CSV.')
if len(csv_names) > 1:
    print('Several CSV files uploaded; using the first one:', csv_names[0])
dataset_path = Path(csv_names[0])
print('Using dataset:', dataset_path, '|', dataset_path.stat().st_size, 'bytes')

## 2. Imports, configuration, loading and dataset audit

In [ ]:
import json, time, platform, shutil
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import sklearn, joblib
from scipy import sparse
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis
from sklearn.covariance import ledoit_wolf
from sklearn.pipeline import Pipeline
from sklearn.model_selection import StratifiedGroupKFold, cross_val_score
from sklearn.metrics import (accuracy_score, precision_score, recall_score, f1_score,
    confusion_matrix, roc_auc_score, average_precision_score, roc_curve,
    precision_recall_curve, ConfusionMatrixDisplay)
from IPython.display import display

OUTPUT_DIR = Path('lda_submission_outputs')
OUTPUT_DIR.mkdir(exist_ok=True)
config = {'target_column': 'class', 'positive_label': -1, 'identifier_columns': ['Index'],
          'test_size': 0.2, 'random_state': 42, 'cv_folds': 5}

df = pd.read_csv(dataset_path)
if config['target_column'] not in df.columns:
    raise ValueError("Expected a target column named 'class'. Check that you uploaded the right CSV.")
if df['class'].isna().any() or set(df['class'].unique()) != {-1, 1}:
    raise ValueError("Expected exactly the labels -1 and +1 in 'class', with no missing labels.")
X = df.drop(columns=[config['target_column'], *config['identifier_columns']], errors='ignore')
y = (df[config['target_column']] == config['positive_label']).astype(int)
if X.isna().any().any():
    raise ValueError('Missing predictor values found; this notebook intentionally does not impute them.')
if not all(pd.api.types.is_numeric_dtype(X[c]) for c in X.columns):
    raise ValueError('This dataset is expected to contain numeric predictors.')

# Group identical feature vectors, even when their labels conflict. This prevents
# an identical vector from appearing in both training and evaluation partitions.
groups, unique_vectors = pd.factorize(pd.MultiIndex.from_frame(X), sort=True)
feature_label_duplicates = int(pd.concat([X, y.rename('target')], axis=1).duplicated().sum())
conflicting_vectors = int(pd.DataFrame({'group': groups, 'target': y.to_numpy()}).groupby('group')['target'].nunique().gt(1).sum())
print('Dataset shape:', df.shape, '| predictors:', X.shape[1])
print('Missing cells:', int(df.isna().sum().sum()))
print('Repeated feature+label rows (Index ignored):', feature_label_duplicates)
print('Unique feature vectors:', len(unique_vectors), '| vectors with conflicting labels:', conflicting_vectors)
print('Feature values:', sorted(np.unique(X.to_numpy()).tolist()))
print('Label coding check (rows = original class, columns = target y):')
display(pd.crosstab(df['class'], y, rownames=['class'], colnames=['y (1 = phishing)']))
display(df.head())


In [ ]:
ax = y.value_counts().sort_index().rename(index={0: 'Legitimate', 1: 'Phishing'}).plot.bar(figsize=(5, 4), rot=0)
ax.set_title('Class distribution'); ax.set_ylabel('Samples')
plt.tight_layout(); plt.savefig(OUTPUT_DIR / 'class_distribution.png', dpi=160); plt.show()

## 3. The mathematics of LDA

Let the two classes be $k\in\{0\ (\text{legitimate}),\,1\ (\text{phishing})\}$ with $n_k$ samples, class means $\mu_k$, overall mean $\mu$, and $d$ features.

**Fisher's criterion (geometric view).** Find a direction $w$ such that, after projecting every sample to the line $w^\top x$, the class means are far apart relative to the spread inside each class:

$$
J(w)=\frac{w^\top S_B\,w}{w^\top S_W\,w},\qquad
S_W=\sum_{k}\sum_{i:\,y_i=k}(x_i-\mu_k)(x_i-\mu_k)^\top,\qquad
S_B=\sum_k n_k(\mu_k-\mu)(\mu_k-\mu)^\top .
$$

Maximising $J$ gives the generalised eigenproblem $S_B w=\lambda S_W w$. With **two classes** $S_B=\frac{n_0n_1}{n}(\mu_1-\mu_0)(\mu_1-\mu_0)^\top$ has rank one, so there is at most **one** nonzero discriminant direction and it has a closed form:

$$
w \;\propto\; S_W^{-1}(\mu_1-\mu_0).
$$

**Probabilistic view (Bayes rule).** If $x\mid k\sim\mathcal N(\mu_k,\Sigma)$ with a **shared** covariance $\Sigma$ and class priors $\pi_k$, the log-odds of "phishing" are *linear* in $x$:

$$
\log\frac{P(y=1\mid x)}{P(y=0\mid x)} = w^\top x + b,\qquad
w=\Sigma^{-1}(\mu_1-\mu_0),\quad
b=-\tfrac12(\mu_1+\mu_0)^\top\Sigma^{-1}(\mu_1-\mu_0)+\log\frac{\pi_1}{\pi_0},
$$

with $\hat\Sigma=S_W/(n-2)$ estimated from the data. Predict *phishing* when $w^\top x+b>0$; the probability is $\sigma(w^\top x+b)$.
Under the shared-covariance model, both views give the **same direction** $w$ (up to scale); the Gaussian view adds the threshold $b$ and posterior probabilities, whose calibration depends on model fit.

**What is assumed, and what for**

| Assumption | Needed for |
|---|---|
| Fisher's direction itself | only means and within-class scatter (no Gaussianity) |
| Gaussian classes, shared $\Sigma$ | the linear rule to be Bayes-optimal and the model posteriors to match the true probabilities |
| $\hat\Sigma$ invertible and stable | computing $w$; collinear features make $S_W^{-1}$ unstable |

**Shrinkage** (solver `lsqr`, `shrinkage='auto'`) replaces $\hat\Sigma$ by $(1-\alpha)\hat\Sigma+\alpha\,\frac{\operatorname{tr}\hat\Sigma}{d}I$ with $\alpha$ chosen by the Ledoit–Wolf formula, stabilising the inverse when features are correlated.

## 4. Preprocessing and grouped stratified train/test split

Standardisation is fitted inside each training fold. We keep every identical feature vector in one group, including vectors that occur with conflicting labels, so copies cannot leak between training, validation and test partitions. The grouped stratified holdout is approximately 80/20; exact size can vary because groups stay intact.


In [ ]:
def make_preprocessor(features):
    numeric = list(features.select_dtypes(include='number').columns)
    categorical = [c for c in features.columns if c not in numeric]
    transformers = []
    if numeric:
        transformers.append(('numeric', StandardScaler(), numeric))
    if categorical:
        transformers.append(('categorical', OneHotEncoder(handle_unknown='ignore'), categorical))
    return ColumnTransformer(transformers, remainder='drop', verbose_feature_names_out=False)

splitter = StratifiedGroupKFold(n_splits=config['cv_folds'], shuffle=True, random_state=config['random_state'])
train_idx, test_idx = next(splitter.split(X, y, groups=groups))
X_train, X_test = X.iloc[train_idx], X.iloc[test_idx]
y_train, y_test = y.iloc[train_idx], y.iloc[test_idx]
groups_train, groups_test = groups[train_idx], groups[test_idx]
assert set(groups_train).isdisjoint(set(groups_test)), 'A feature group crossed the train/test split.'
print('Training rows:', len(X_train), '| test rows:', len(X_test))
print('Unique feature groups train:', len(np.unique(groups_train)), '| test:', len(np.unique(groups_test)))
print('Phishing share train: %.4f | test: %.4f' % (y_train.mean(), y_test.mean()))


## 5. Compare two LDA configurations (grouped training-only 5-fold CV, metric = F1)

The CV folds also keep identical feature vectors together. Only training rows contribute to model selection.


In [ ]:
candidates = {
    'svd': LinearDiscriminantAnalysis(solver='svd'),
    'lsqr_auto': LinearDiscriminantAnalysis(solver='lsqr', shrinkage='auto'),
}
cv = StratifiedGroupKFold(n_splits=config['cv_folds'], shuffle=True, random_state=config['random_state'])
comparison = []
for name, estimator in candidates.items():
    pipe = Pipeline([('preprocess', make_preprocessor(X_train)), ('lda', estimator)])
    scores = cross_val_score(pipe, X_train, y_train, groups=groups_train, cv=cv, scoring='f1', n_jobs=1)
    comparison.append({'configuration': name, 'cv_f1_mean': float(scores.mean()), 'cv_f1_std': float(scores.std())})
comparison_df = pd.DataFrame(comparison)
selected_name = comparison_df.loc[comparison_df['cv_f1_mean'].idxmax(), 'configuration']
print('Selected LDA configuration:', selected_name)
display(comparison_df)
comparison_df.to_csv(OUTPUT_DIR / 'lda_comparison.csv', index=False)
ax = comparison_df.plot.bar(x='configuration', y='cv_f1_mean', legend=False, figsize=(5, 4), rot=0)
ax.set_ylabel('Mean grouped CV F1'); ax.set_xlabel('LDA configuration')
plt.tight_layout(); plt.savefig(OUTPUT_DIR / 'lda_comparison.png', dpi=160); plt.show()


## 6. Train the selected model and evaluate once on the held-out test set

In [ ]:
model = Pipeline([('preprocess', make_preprocessor(X_train)), ('lda', candidates[selected_name])])
t0 = time.perf_counter(); model.fit(X_train, y_train); training_seconds = time.perf_counter() - t0
t0 = time.perf_counter()
y_pred = model.predict(X_test); y_prob = model.predict_proba(X_test)[:, 1]
inference_seconds = time.perf_counter() - t0
tn, fp, fn, tp = confusion_matrix(y_test, y_pred, labels=[0, 1]).ravel()
metrics = {'accuracy': float(accuracy_score(y_test, y_pred)),
           'precision': float(precision_score(y_test, y_pred)),
           'recall': float(recall_score(y_test, y_pred)),
           'f1': float(f1_score(y_test, y_pred)),
           'specificity': float(tn / (tn + fp)),
           'roc_auc': float(roc_auc_score(y_test, y_prob)),
           'pr_auc': float(average_precision_score(y_test, y_prob))}
results = {
    'dataset': {'rows': len(df), 'columns': len(df.columns), 'predictors': X.shape[1], 'test_samples': len(y_test),
                'class_counts': {'phishing': int(y.sum()), 'legitimate': int((1 - y).sum())}},
    'metrics': {**metrics, 'confusion_matrix': {'tn': int(tn), 'fp': int(fp), 'fn': int(fn), 'tp': int(tp)}},
    'timing': {'training_seconds': training_seconds, 'test_inference_seconds': inference_seconds,
               'test_samples': len(y_test), 'mean_ms_per_sample': inference_seconds / len(y_test) * 1000,
               'throughput_samples_per_second': len(y_test) / inference_seconds},
    'selection': {'candidates': comparison, 'selected': selected_name, 'cv': '5-fold stratified grouped training-only F1'},
    'split': {'method': 'StratifiedGroupKFold', 'folds': config['cv_folds'], 'random_state': config['random_state'],
              'group_key': 'identical predictor vectors', 'train_samples': len(y_train), 'test_samples': len(y_test),
              'train_unique_groups': int(len(np.unique(groups_train))), 'test_unique_groups': int(len(np.unique(groups_test)))},
    'environment': {'python': platform.python_version(), 'scikit_learn': sklearn.__version__, 'platform': platform.platform()},
    'positive_class': 'phishing (-1 mapped to 1)'}
display(pd.DataFrame([metrics]).T.rename(columns={0: 'Score'}))
print('Confusion matrix  TN=%d FP=%d FN=%d TP=%d' % (tn, fp, fn, tp))
(OUTPUT_DIR / 'results.json').write_text(json.dumps(results, indent=2))
pd.DataFrame([{**metrics, **results['timing']}]).to_csv(OUTPUT_DIR / 'results.csv', index=False)

In [ ]:
fig, ax = plt.subplots(figsize=(5, 4))
ConfusionMatrixDisplay(np.array([[tn, fp], [fn, tp]]), display_labels=['Legitimate', 'Phishing']).plot(ax=ax, cmap='Blues', colorbar=False)
ax.set_title('Confusion matrix (test set)'); plt.tight_layout(); plt.savefig(OUTPUT_DIR / 'confusion_matrix.png', dpi=160); plt.show()

fpr, tpr, _ = roc_curve(y_test, y_prob)
plt.figure(figsize=(5.5, 4)); plt.plot(fpr, tpr, label=f'ROC-AUC = {metrics["roc_auc"]:.4f}'); plt.plot([0, 1], [0, 1], '--', label='Chance')
plt.xlabel('False positive rate'); plt.ylabel('True positive rate'); plt.title('ROC curve'); plt.legend()
plt.tight_layout(); plt.savefig(OUTPUT_DIR / 'roc_curve.png', dpi=160); plt.show()

pv, rv, _ = precision_recall_curve(y_test, y_prob)
plt.figure(figsize=(5.5, 4)); plt.plot(rv, pv, label=f'PR-AUC = {metrics["pr_auc"]:.4f}')
plt.xlabel('Recall'); plt.ylabel('Precision'); plt.title('Precision-recall curve'); plt.legend()
plt.tight_layout(); plt.savefig(OUTPUT_DIR / 'precision_recall.png', dpi=160); plt.show()

# Part C: Understanding LDA

So far LDA was a black box. Now we rebuild it from the formulas in Section 3 using only NumPy and compare with scikit-learn.

## 7. LDA from scratch
We work in the standardised feature space $Z$ that the fitted preprocessor produces, compute $\mu_0,\mu_1,S_W,S_B$, then $w$ and $b$, and compare with scikit-learn's `coef_` and `intercept_`.

In [ ]:
pre = model.named_steps['preprocess']
Z_train, Z_test = pre.transform(X_train), pre.transform(X_test)
if sparse.issparse(Z_train):
    Z_train, Z_test = Z_train.toarray(), Z_test.toarray()
feat_names = list(pre.get_feature_names_out())
ytr = y_train.to_numpy(); yte = y_test.to_numpy()

Z0, Z1 = Z_train[ytr == 0], Z_train[ytr == 1]
n0, n1 = len(Z0), len(Z1); n = n0 + n1; d = Z_train.shape[1]
mu0, mu1, mu = Z0.mean(0), Z1.mean(0), Z_train.mean(0)

S_W = (Z0 - mu0).T @ (Z0 - mu0) + (Z1 - mu1).T @ (Z1 - mu1)                       # within-class scatter
S_B = n0 * np.outer(mu0 - mu, mu0 - mu) + n1 * np.outer(mu1 - mu, mu1 - mu)       # between-class scatter
print('rank(S_B) =', np.linalg.matrix_rank(S_B), ' (rank 1 here; at most 1 nonzero direction for two classes)')

w_fisher = np.linalg.pinv(S_W) @ (mu1 - mu0)           # Fisher direction  S_W^{-1}(mu1 - mu0)
w_lda = (n - 2) * w_fisher                             # = Sigma_hat^{-1}(mu1 - mu0), Sigma_hat = S_W/(n-2)
b_lda = -0.5 * w_lda @ (mu0 + mu1) + np.log(n1 / n0)  # threshold incl. log prior ratio

scores_hand = Z_test @ w_lda + b_lda
pred_hand = (scores_hand > 0).astype(int)

lda_svd = LinearDiscriminantAnalysis(solver='svd').fit(Z_train, ytr)
lda_sel = model.named_steps['lda']

def cosine(a, b):
    return float(a @ b / (np.linalg.norm(a) * np.linalg.norm(b)))

rows = []
for name, ref in [('sklearn svd', lda_svd), (f'selected model ({selected_name})', lda_sel)]:
    pred_ref = ref.predict(Z_test)
    rows.append({'compared with': name,
                 'cosine(w_hand, coef_)': cosine(w_lda, ref.coef_.ravel()),
                 'max |w_hand - coef_|': float(np.abs(w_lda - ref.coef_.ravel()).max()),
                 '|b_hand - intercept_|': float(abs(b_lda - ref.intercept_[0])),
                 'test predictions identical (%)': 100 * float((pred_hand == pred_ref).mean())})
display(pd.DataFrame(rows).set_index('compared with'))
print('Hand-built LDA on the test set  ->  accuracy %.4f, F1 %.4f' % (accuracy_score(yte, pred_hand), f1_score(yte, pred_hand)))

The hand-built rule should agree with scikit-learn's `svd` solver to numerical precision (cosine ≈ 1, ~100 % identical predictions). Agreement with the shrinkage model is slightly lower because shrinkage deliberately changes $\hat\Sigma$ and therefore $w$.

## 8. Does $w=S_W^{-1}(\mu_1-\mu_0)$ really maximise Fisher's $J$?
We compare $J(w)$ for (i) the closed-form direction, (ii) each single feature taken alone, and (iii) 20 000 random directions. The random search is an illustration, not a proof of the maximum.

In [ ]:
def fisher_J(w):
    return float((w @ S_B @ w) / (w @ S_W @ w))

J_opt = fisher_J(w_fisher)
J_axes = np.array([fisher_J(e) for e in np.eye(d)])
rng = np.random.default_rng(config['random_state'])
W = rng.normal(size=(20000, d))
J_rand = np.einsum('ij,jk,ik->i', W, S_B, W) / np.einsum('ij,jk,ik->i', W, S_W, W)
best_axis = int(J_axes.argmax())

print('J(Fisher direction)        : %.5f' % J_opt)
print('J(sklearn svd coef_)       : %.5f' % fisher_J(lda_svd.coef_.ravel()))
print('best single feature (%s) : %.5f' % (feat_names[best_axis], J_axes[best_axis]))
print('best of 20000 random dirs  : %.5f' % J_rand.max())
print('Fisher direction is the maximum:', bool(J_opt >= max(J_axes.max(), J_rand.max()) - 1e-9))

fig, ax = plt.subplots(figsize=(6.5, 4))
ax.hist(J_rand, bins=60, color='#9db4d6', label='random directions')
ax.axvline(J_axes.max(), color='orange', ls='--', label='best single feature')
ax.axvline(J_opt, color='crimson', lw=2, label='Fisher direction')
ax.set_xlabel('Fisher criterion J(w)'); ax.set_ylabel('Count'); ax.set_title('Fisher criterion: closed-form direction vs. alternatives')
ax.legend(); plt.tight_layout(); plt.savefig(OUTPUT_DIR / 'fisher_criterion.png', dpi=160); plt.show()

## 9. The one-dimensional discriminant projection
LDA reduces each website to a *single number* $s=w^\top x+b$. The two classes' histograms of $s$ on the **test** set show how well one line separates them; the decision threshold is $s=0$. We also report the Mahalanobis distance $D=\sqrt{(\mu_1-\mu_0)^\top\hat\Sigma^{-1}(\mu_1-\mu_0)}$ (training data). Under the Gaussian/shared-covariance model with equal priors the Bayes error would be $\Phi(-D/2)$, a benchmark to compare with the observed error.

In [ ]:
from scipy.stats import norm
s_sel = model.decision_function(X_test)

fig, axes = plt.subplots(1, 2, figsize=(11, 4), sharey=True)
for ax, s, title in [(axes[0], s_sel, f'scikit-learn ({selected_name})'), (axes[1], scores_hand, 'hand-built NumPy LDA')]:
    bins = np.linspace(np.percentile(s, 0.5), np.percentile(s, 99.5), 50)
    ax.hist(s[yte == 0], bins=bins, alpha=0.65, label='Legitimate', color='tab:green')
    ax.hist(s[yte == 1], bins=bins, alpha=0.65, label='Phishing', color='tab:red')
    ax.axvline(0, color='k', ls='--', label='threshold s = 0')
    ax.set_title(title); ax.set_xlabel('discriminant score  w·x + b')
axes[0].set_ylabel('Test samples'); axes[0].legend()
plt.tight_layout(); plt.savefig(OUTPUT_DIR / 'discriminant_projection.png', dpi=160); plt.show()

D = float(np.sqrt((mu1 - mu0) @ (w_lda)))          # w_lda = Sigma_hat^{-1}(mu1-mu0)
print('Mahalanobis distance D = %.3f' % D)
print('Theoretical Bayes error under Gaussian/equal-cov/equal-prior model: %.4f' % norm.cdf(-D / 2))
print('Observed test error of the selected model                       : %.4f' % (1 - metrics['accuracy']))

## 10. Checking LDA's assumptions on the dataset
LDA's *optimality* assumes Gaussian classes with a shared covariance. We check three things: (a) are the features Gaussian at all, (b) are the class covariances equal, (c) are the features strongly collinear.

In [ ]:
# (a) feature value distributions
vals = sorted(np.unique(X.to_numpy()).tolist())
prop = pd.DataFrame({v: (X == v).mean() for v in vals})
print('Distinct values per feature: min %d, max %d' % (X.nunique().min(), X.nunique().max()))
print('Features with <= 3 distinct values: %d of %d' % ((X.nunique() <= 3).sum(), X.shape[1]))
ax = prop.plot.bar(stacked=True, figsize=(11, 4), width=0.85)
ax.set_ylabel('Share of samples'); ax.set_title('Value distribution of every predictor (discrete, clearly non-Gaussian)')
ax.legend(title='value', bbox_to_anchor=(1.01, 1), loc='upper left')
plt.tight_layout(); plt.savefig(OUTPUT_DIR / 'feature_value_distributions.png', dpi=160); plt.show()

In [ ]:
# (b) equal class covariances?
C0, C1 = np.cov(Z0, rowvar=False), np.cov(Z1, rowvar=False)
C_pool = (n0 - 1) / (n - 2) * C0 + (n1 - 1) / (n - 2) * C1
rel_diff = np.linalg.norm(C1 - C0) / np.linalg.norm(C_pool)
print('Relative difference ||C1 - C0||_F / ||C_pooled||_F = %.3f   (0 = identical covariances)' % rel_diff)
var_ratio = pd.Series(np.diag(C1) / np.diag(C0), index=feat_names).sort_values()
print('Per-feature variance ratio phishing/legitimate: min %.2f, median %.2f, max %.2f' % (var_ratio.min(), var_ratio.median(), var_ratio.max()))

fig, axes = plt.subplots(1, 3, figsize=(14, 4))
lim = max(np.abs(C0).max(), np.abs(C1).max())
for ax, M, t in [(axes[0], C0, 'Covariance: legitimate'), (axes[1], C1, 'Covariance: phishing')]:
    im = ax.imshow(M, cmap='coolwarm', vmin=-lim, vmax=lim); ax.set_title(t); plt.colorbar(im, ax=ax, fraction=0.046)
dl = np.abs(C1 - C0).max()
im = axes[2].imshow(C1 - C0, cmap='coolwarm', vmin=-dl, vmax=dl); axes[2].set_title('Difference (phishing - legitimate)'); plt.colorbar(im, ax=axes[2], fraction=0.046)
plt.tight_layout(); plt.savefig(OUTPUT_DIR / 'class_covariances.png', dpi=160); plt.show()

# (c) collinearity
R = np.corrcoef(Z_train, rowvar=False)
iu = np.triu_indices(d, 1)
print('Feature pairs with |r| > 0.5: %d,  |r| > 0.8: %d  (of %d pairs)' % ((np.abs(R[iu]) > 0.5).sum(), (np.abs(R[iu]) > 0.8).sum(), len(iu[0])))
print('Condition number of pooled covariance: %.1f' % np.linalg.cond(C_pool))

**What this means for our data**

Every feature in this dataset takes only a few fixed values (at most three, like -1, 0, 1). It is not a smooth bell-shaped curve, so the "Gaussian" assumption of LDA is not true for our data.

LDA also assumes both classes have the same spread. The covariance difference and variance ratios printed above tell us how true that is. If the difference is small, the assumption is reasonable. If it is large, a model that allows each class its own spread (QDA) might be better.

Even so, LDA can still work well, for three reasons:
1. Finding the best separating direction only needs the class averages and the spread inside each class. It does not need bell-shaped data. If the two classes have clearly different averages, it works.
2. Using many useful indicators together can give an accurate straight-line boundary, even when the ideal conditions are not met.
3. We have thousands of samples but only 30 features, so the spread is estimated reliably.

The downside is that the **probabilities** from `predict_proba` are less trustworthy than the yes/no decisions. Treat them as scores, not exact chances.

Finally, compare the real test error with the theoretical error printed in Section 9 and say how close they are.

## 11. What does shrinkage do to the covariance estimate?

LDA needs the "spread" of the data (the covariance matrix), and it has to invert it. When some features move together, the inversion becomes unstable and small errors get blown up.

Shrinkage fixes this by pulling the estimated spread slightly towards a simple, evenly spread version of itself. The amount of pulling is called α. If α is 0, nothing changes. If α is 1, we use only the simple version. Scikit-learn picks α automatically (the Ledoit–Wolf method) and combines the two classes using their sizes.

In this section we:
1. Rebuild this calculation ourselves and check it matches scikit-learn.
2. Compare the eigenvalues (the "strength" of each direction of spread) before and after shrinkage. Shrinkage raises the smallest ones, and those are the ones that cause the instability.

We have thousands of samples but only 30 features, so the spread is already estimated well. We therefore expect α to be small and the result to be almost the same as plain LDA. The cross-validation table in Section 5 shows whether shrinkage actually helps on this data.

In [ ]:
def shrunk_class_cov(Xg):
    # replicates scikit-learn's _cov(X, shrinkage='auto')
    sc = StandardScaler().fit(Xg)
    s_cov, alpha = ledoit_wolf(sc.transform(Xg))
    return sc.scale_[:, None] * s_cov * sc.scale_[None, :], alpha

pi0, pi1 = n0 / n, n1 / n
(S0, a0), (S1, a1) = shrunk_class_cov(Z0), shrunk_class_cov(Z1)
Sigma_shrunk = pi0 * S0 + pi1 * S1
Sigma_plain = pi0 * np.cov(Z0, rowvar=False, bias=True) + pi1 * np.cov(Z1, rowvar=False, bias=True)

lda_lsqr = LinearDiscriminantAnalysis(solver='lsqr', shrinkage='auto').fit(Z_train, ytr)
print('Replicated shrunk covariance equals scikit-learn covariance_:', bool(np.allclose(Sigma_shrunk, lda_lsqr.covariance_, atol=1e-8)))
print('Shrinkage intensity alpha  legitimate: %.4f   phishing: %.4f' % (a0, a1))

ev_plain = np.sort(np.linalg.eigvalsh(Sigma_plain))[::-1]
ev_shr = np.sort(np.linalg.eigvalsh(Sigma_shrunk))[::-1]
print('Smallest eigenvalue   plain: %.3e   shrunk: %.3e' % (ev_plain[-1], ev_shr[-1]))
print('Condition number      plain: %.1f   shrunk: %.1f' % (ev_plain[0] / max(ev_plain[-1], 1e-300), ev_shr[0] / ev_shr[-1]))

plt.figure(figsize=(6.5, 4))
plt.semilogy(ev_plain, 'o-', label='plain pooled covariance'); plt.semilogy(ev_shr, 's-', label='Ledoit-Wolf shrunk covariance')
plt.xlabel('Eigenvalue rank'); plt.ylabel('Eigenvalue (log scale)'); plt.title('Effect of shrinkage on the covariance spectrum'); plt.legend()
plt.tight_layout(); plt.savefig(OUTPUT_DIR / 'shrinkage_spectrum.png', dpi=160); plt.show()

print('Cosine between plain-LDA and shrinkage-LDA directions: %.5f' % cosine(lda_svd.coef_.ravel(), lda_lsqr.coef_.ravel()))

# Part D: Interpretation, outputs and notes

## 12. Coefficients of the selected model
The coefficients live in *standardised* feature space, so magnitudes are comparable across features. Positive means "pushes towards phishing", negative "towards legitimate". They are descriptive weights of a correlated linear model, **not** proof of causal importance.

In [ ]:
coefficients = np.ravel(model.named_steps['lda'].coef_)
coefficient_df = pd.DataFrame({'feature': pre.get_feature_names_out(), 'coefficient': coefficients})
coefficient_df['abs_coefficient'] = coefficient_df['coefficient'].abs()
coefficient_df = coefficient_df.sort_values('abs_coefficient', ascending=False)
display(coefficient_df.head(15))
coefficient_df.to_csv(OUTPUT_DIR / 'coefficients.csv', index=False)

top = coefficient_df.head(15).iloc[::-1]
plt.figure(figsize=(7, 5)); plt.barh(top['feature'], top['coefficient'], color=np.where(top['coefficient'] > 0, 'tab:red', 'tab:green'))
plt.xlabel('LDA coefficient (standardised space)'); plt.title('Top 15 features (red = towards phishing)')
plt.tight_layout(); plt.savefig(OUTPUT_DIR / 'top_coefficients.png', dpi=160); plt.show()

joblib.dump({'pipeline': model, 'features': list(X.columns), 'positive_label': -1}, OUTPUT_DIR / 'lda_pipeline.joblib')

## 13. Predict a single record with the saved model

In [ ]:
def predict_record(record, bundle):
    feats = bundle['features']
    missing, extra = set(feats) - set(record), set(record) - set(feats)
    if missing: raise ValueError('Missing features: ' + ', '.join(sorted(missing)))
    if extra: raise ValueError('Unexpected features: ' + ', '.join(sorted(extra)))
    row = pd.DataFrame([{k: record[k] for k in feats}], columns=feats)
    p = bundle['pipeline'].predict_proba(row)[0]
    return {'prediction': 'phishing' if p[1] > 0.5 else 'legitimate',
            'phishing_probability': float(p[1]), 'legitimate_probability': float(p[0])}

bundle = joblib.load(OUTPUT_DIR / 'lda_pipeline.joblib')
for i in [0, 1, 2]:
    rec = X_test.iloc[i].to_dict()
    print('test row', i, '| true:', 'phishing' if yte[i] else 'legitimate', '|', predict_record(rec, bundle))

## 14. Package the outputs

In [ ]:
archive = shutil.make_archive('lda_submission_outputs', 'zip', root_dir=OUTPUT_DIR)
print('Saved:', archive)
try:
    from google.colab import files
    files.download(archive)
except Exception as e:
    print('Download skipped:', e)

## 15. Notes and limitations
- **Data:** the CSV contains pre-computed numeric indicators (not raw URLs), so the model never inspects a live website. Results apply to this fixed benchmark snapshot.
- **Preprocessing:** only standardisation is used. The dataset has no missing values, so no imputation step is needed.
- **Evaluation:** a grouped random split cannot show robustness and does not group distinct feature vectors by domain to new time periods or unseen domains; features may embed external reputation signals.
- **Probabilities:** `predict_proba` assumes Gaussian classes with shared covariance, which these discrete features violate, so treat probabilities as scores rather than calibrated risks.
- **Timing:** measured locally in memory and depends on the Colab hardware; it is not a production latency figure.
- **Coefficients:** descriptive linear weights, not causal importance.
- **Reproducibility:** `random_state = 42` everywhere; library versions are stored in `results.json`.